[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/15-guided-purchase.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 15. Guided purchase

You are the customer. The seller stays in one chat until you type bye.

Ask for a jar you can eat, one of those jars in a cart to Ohio, a check of that cart, and an order note. The same message list is the visit. When `file_order_note` waits, your next message approves it: "yes, file the note", or the code the seller shows. The seller calls the tool again on this thread. No card is charged.

A 1000-jar cart is still rejected before any model call. The page from tutorial 10 still cannot open the canary. `charge_card` stays denied.


## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Checker and gate, before the model

A bad quantity is rejected. The order note is unwritten until the token matches. The charge stays denied.

In [ ]:
import json

import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.memory
import tutorial.common.skills
import tutorial.common.verify
import tutorial.common.gate
import tutorial.common.plan
import tutorial.common.trace
import tutorial.common.cost

from tutorial.common.autonomy import approval_token
from tutorial.common.cost import cost_summary, reset_cache
from tutorial.common.gate import ORDERS, gate_call, reset_orders
from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.injection import show_injection_boundaries
from tutorial.common.tools import call_tool
from tutorial.common.verify import verify_cart

reset_db()
reset_orders()
reset_cache()
bad = json.loads(verify_cart({
    "sku": "HJ-CHI", "qty": 1000, "ship": True, "destination": "Ohio", "citation": "origin note",
}))
print("bad cart", bad)
check(bad["status"] == "rejected", "a 1000-jar cart is rejected")
args = {
    "customer": "Maya",
    "sku": "HJ-CHI",
    "qty": 1,
    "ship": True,
    "destination": "Ohio",
    "citation": "catalog HJ-CHI",
}
held = gate_call("file_order_note", args, set())
check(held["decision"] == "confirm_required", "the order note waits for a person")
check(not list(ORDERS.glob("*.md")), "nothing is written before approval")
allowed = gate_call("file_order_note", args, {held["approval_token"]})
check(allowed["decision"] == "allowed", "the same note is allowed when the token matches")
print(call_tool({"name": "file_order_note", "arguments": args}))
note = next(ORDERS.glob("*.md")).read_text()
print(note)
check("charged: no" in note, "the note does not charge a card")
charge = gate_call(
    "charge_card",
    {"amount_cents": 1800},
    {approval_token("charge_card", {"amount_cents": 1800})},
)
check(charge["decision"] == "denied", "a card charge stays denied")

## The page still cannot grant a tool

Same boundary as tutorial 10. The canary stays out of the output.

In [ ]:
show_injection_boundaries()

## One visit, one message list

`run_customer_chat`, defined in the next cell, is the customer visit. It is not a second agent loop. Tutorial 2 still answers one line: `run_turn` calls the model, runs tools, and stops. This function only decides what it means to stay with one customer at Harbor Jar until they leave.

The small helpers in that cell match a goodbye, a yes, and a live prompt. They sit in the same cell because they are awkward on their own. The comment immediately above `run_customer_chat` is the reading order. Follow it while you read the body.

1. **Greet.** The seller prints a welcome. The system message is the lesson text plus a short note: this list is one customer, earlier lines already happened, `file_order_note` waits until a later message agrees, and `charge_card` stays denied. No card is charged in this chat.

2. **Read lines until bye.** When a person can type, each line is `input("You: ")`. The visit stops when the whole line is `bye`, `quit`, `exit`, or `goodbye`, or when input ends. When nobody can type, the same function replays `demo_utterances` and prints each line as `You:`. Jupyter, VS Code, and Colab count as a person. A headless run such as `nbconvert` does not, so the visit can finish without a terminal.

3. **Append the user message.** A non-empty line that is not a goodbye is appended to the message list started at the greeting. The next turn sees the cart, the held note, and the yes, because the list is the same object.

4. **Plan once.** The first real line calls `attach_plan`, the hook from tutorial 14. The plan is written onto the system message. Later lines do not draft a second plan. If that hook is off, the plan is empty and the system message stays as it was.

5. **Detect a yes for a pending confirm token.** If a confirm token is still waiting, this line may approve it. Pasting the token approves that call. "Yes", "yes, file the note", and the same kind of reply approve every call still waiting. A bare no, or a refusal to file, approves nothing. A token that was never waiting is ignored, so a sentence cannot grant `charge_card`.

6. **Call `run_turn` on that same list, with `max_steps`.** `max_steps` is how many model calls, and therefore how many tool steps, one customer line may take. It is not how many rounds the conversation may last. The visit continues until bye, and each line gets its own budget. Confirmed tokens are passed in so the gate can allow `file_order_note` with the same arguments and still deny a charge.

7. **Print the seller's reply.** The tool log, the spans, and the usage rows stay on the visit. A new confirm token is remembered as pending. If one is still waiting, the seller also prints the code to send back.

8. **Continue.** The next line follows the same steps. After bye, the seller says goodbye and returns the visit: the mode, the message list, the turn count, the last reply, the plan, the tool log, the spans, the usage, and the tokens the customer approved.

In [ ]:
import re
import sys

from tutorial.common.loop import attach_plan, run_turn

WELCOME = (
    "Welcome to Harbor Jar. I'm the seller on the website. "
    "Tell me what you'd like. I can recommend a jar, check a cart, "
    "and file an order note after you say yes. "
    "No card is charged. Type bye when you want to leave."
)
FAREWELL = "Bye. No card was charged from this chat."

SESSION_NOTE = (
    "\n\nThis is one visit with one customer. "
    "Earlier messages in this list already happened. Continue from them. "
    "Recommend a jar, put it in a cart, call verify_cart, then file_order_note. "
    "file_order_note waits until the customer agrees in a later message. "
    "On that waiting turn, ask them to say yes. "
    "The note is filed only when the tool result says WROTE. "
    "charge_card stays denied. No card is charged in this chat."
)
_APPROVED_MARK = "The customer has approved the order note that was waiting."
APPROVED_NOTE = (
    "\n\n" + _APPROVED_MARK + " "
    "Call file_order_note again with the same arguments. "
    "Leave charge_card alone."
)

_BYE = {"bye", "quit", "exit", "goodbye"}
_YES = re.compile(
    r"("
    r"^\s*(yes|yep|yeah|sure|ok|okay|approve|approved|confirm|confirmed)\b"
    r"|\byes\b"
    r"|\bapprove[d]?\b"
    r"|\bconfirm(?:ed)?\b"
    r"|\bgo ahead\b"
    r"|\bplease file\b"
    r"|\bfile (?:the |that |my )?note\b"
    r"|\bfile it\b"
    r")",
    re.IGNORECASE,
)
_ONLY_NO = re.compile(r"^\s*(no|nope|nah)[.!]?\s*$", re.IGNORECASE)
_REFUSE_NOTE = re.compile(
    r"("
    r"\bdon'?t file\b"
    r"|\bdo not file\b"
    r"|\bdon'?t write\b"
    r"|\bdo not write\b"
    r"|\bnever mind\b"
    r"|\bnevermind\b"
    r")",
    re.IGNORECASE,
)


def is_bye(text):
    """True when the whole line is bye, quit, exit, or goodbye."""
    word = (text or "").strip().lower().strip("!.,;: ")
    return word in _BYE


def approved_tokens(text, pending):
    """Return confirm tokens this customer line approves.

    A pasted token approves that waiting call. "Yes", "yes, file the note",
    and the same kind of reply approve every call still waiting. A bare no,
    or a refusal to file, approves nothing. A token that was never waiting
    is ignored, so a card charge cannot be granted from the sentence.
    """
    waiting = [token for token in (pending or []) if token]
    if not text or not waiting:
        return []
    lowered = text.lower()
    pasted = [token for token in waiting if token.lower() in lowered]
    if pasted:
        return pasted
    if _ONLY_NO.search(text) or _REFUSE_NOTE.search(text):
        return []
    if _YES.search(text):
        return list(waiting)
    return []


def prompts_a_person():
    """True when ``input`` reaches a person.

    A terminal is the plain case. Jupyter, VS Code, and Colab also prompt,
    even though their stdin is not a TTY. ``nbconvert`` sets the kernel's
    stdin flag off, and a pipe has no kernel. Those replay ``demo_utterances``
    so the visit can finish.
    """
    stdin = sys.stdin
    if stdin is not None and getattr(stdin, "isatty", lambda: False)():
        return True
    shell = _ipython()
    if shell is None:
        return False
    allowed = _kernel_allow_stdin(shell)
    if allowed is not None:
        return allowed
    name = type(shell).__name__.lower()
    if "terminal" in name:
        return False
    return True


# How run_customer_chat runs one Harbor Jar visit.
#
# The loop from tutorial 2 still handles a single customer line. This
# function is only the visit around that loop. Read the body in this order.
#
# 1. Greet.
#    Print the Harbor Jar welcome. The system message is the lesson text
#    plus a note that this list is one customer, that file_order_note waits
#    for a later yes, and that charge_card stays denied.
#
# 2. Read lines until bye.
#    Interactive mode calls input("You: ") until the whole line is bye,
#    quit, exit, or goodbye, or until input ends. Demo mode is the fallback
#    when nobody can type (nbconvert, a pipe). It replays demo_utterances
#    and prints each line as You: so the transcript looks the same.
#
# 3. Append the user message.
#    A non-empty line that is not a goodbye is appended to the same
#    message list. The list is not rebuilt between turns, so a later yes
#    still sees the cart and the held note.
#
# 4. Plan once.
#    On the first real line, attach_plan writes the tutorial 14 plan onto
#    the system message. Later lines leave that plan where it is. No plan
#    hook means an empty plan and an unchanged system message.
#
# 5. Detect a yes for a pending confirm token.
#    A pasted token approves that waiting call. "Yes", "yes, file the
#    note", and the same kind of reply approve every call still waiting.
#    A bare no, or a refusal to file, approves nothing. A token that was
#    never waiting is ignored, so the sentence cannot grant charge_card.
#
# 6. Call run_turn on this same message list, with max_steps.
#    max_steps is the number of model calls, and therefore tool steps,
#    allowed for this one line. It is not a cap on how many rounds the
#    visit may last. The visit lasts until bye. Each line gets its own
#    budget. Confirmed tokens go in so the gate can allow file_order_note
#    and still deny charge_card.
#
# 7. Print the seller reply.
#    The tool log, the spans, and the usage rows accumulate on the visit.
#    A new confirm token is kept as pending. If one is still waiting, the
#    seller also prints the code to send.
#
# 8. Continue.
#    The next line is read the same way. After bye, print the farewell
#    and return the visit: messages, last reply, plan, tool log, spans,
#    usage, and the tokens the customer approved.
def run_customer_chat(
    system,
    demo_utterances=None,
    max_steps=8,
    trace_id="guided-purchase",
):
    """Greet the customer and talk until they leave.

    The primary path reads ``input("You: ")``. When nobody can type, the
    lines in ``demo_utterances`` are replayed instead. Returns the visit:
    the same message list, the seller's last reply, tool log, spans, usage,
    plan, and the confirm tokens the customer approved along the way.
    """
    mode = "interactive" if prompts_a_person() else "demo"
    messages = [{"role": "system", "content": (system or "").rstrip() + SESSION_NOTE}]
    usage = []
    spans = []
    tool_log = []
    pending = []
    confirmed = set()
    plan = ""
    planned = False
    last_text = ""
    turns = 0
    if mode == "demo" and not demo_utterances:
        print(
            "No terminal is attached, and no demo lines were given. The visit ends here."
        )
        return _visit(
            mode, messages, turns, last_text, plan, tool_log, spans, usage, pending, confirmed
        )
    if mode == "demo":
        print(
            "No terminal is attached, so this visit replays a short list of customer lines."
        )
        print()
    print("Seller: " + WELCOME)
    print(flush=True)
    for raw in _lines(mode, demo_utterances):
        text = (raw or "").strip()
        if not text:
            continue
        if is_bye(text):
            break
        messages.append({"role": "user", "content": text})
        turns += 1
        print()
        print("customer turn " + str(turns))
        if not planned:
            plan = attach_plan(messages, text, usage)
            planned = True
        approved = approved_tokens(text, pending)
        if approved:
            confirmed.update(approved)
            pending = [token for token in pending if token not in confirmed]
            _record_approval(messages)
            print("approved: " + " ".join(approved))
        turn_trace = trace_id if turns == 1 else trace_id + "-" + str(turns)
        result = run_turn(
            messages,
            max_steps=max_steps,
            confirmed_tokens=confirmed,
            trace_id=turn_trace,
            spans=spans,
            usage_rows=usage,
            plan=plan,
        )
        tool_log.extend(result["tool_log"])
        for token in result["pending_tokens"]:
            if token not in confirmed and token not in pending:
                pending.append(token)
        last_text = result["text"]
        print()
        print("Seller: " + last_text)
        if pending:
            print("Seller: " + _waiting_line(pending))
        print(flush=True)
    print()
    print("Seller: " + FAREWELL)
    print("visit: turns=" + str(turns) + " mode=" + mode)
    return _visit(
        mode, messages, turns, last_text, plan, tool_log, spans, usage, pending, confirmed
    )


def _visit(mode, messages, turns, text, plan, tool_log, spans, usage, pending, confirmed):
    return {
        "mode": mode,
        "messages": messages,
        "turns": turns,
        "text": text,
        "plan": plan,
        "tool_log": tool_log,
        "spans": spans,
        "usage": usage,
        "pending_tokens": list(pending),
        "confirmed_tokens": sorted(confirmed),
    }


def _lines(mode, demo_utterances):
    if mode == "interactive":
        return _input_lines()
    return _demo_lines(demo_utterances)


def _input_lines():
    while True:
        try:
            yield input("You: ")
        except EOFError:
            return
        except Exception as error:
            if type(error).__name__ == "StdinNotImplementedError":
                return
            raise


def _demo_lines(demo_utterances):
    for line in demo_utterances or []:
        print("You: " + line)
        yield line


def _record_approval(messages):
    content = messages[0]["content"]
    if _APPROVED_MARK in content:
        return
    messages[0]["content"] = content + APPROVED_NOTE


def _waiting_line(pending):
    codes = ", ".join(pending)
    return (
        "I still need a yes before I do that. "
        "Reply yes, or send this code: " + codes + "."
    )


def _ipython():
    try:
        from IPython import get_ipython
    except ImportError:
        return None
    try:
        return get_ipython()
    except Exception:
        return None


def _kernel_allow_stdin(shell):
    kernel = getattr(shell, "kernel", None)
    if kernel is not None and hasattr(kernel, "_allow_stdin"):
        return bool(kernel._allow_stdin)
    parent = None
    getter = getattr(shell, "get_parent", None)
    if callable(getter):
        try:
            parent = getter()
        except Exception:
            parent = None
    if isinstance(parent, dict):
        content = parent.get("content")
        if isinstance(content, dict) and "allow_stdin" in content:
            return bool(content["allow_stdin"])
    return None


## Talk with the seller

`run_customer_chat` is the visit defined above. This cell starts it.

The seller greets you, then reads `You:` until you type bye, quit, or exit. A yes on a later line approves the note that is waiting. The message list is the same one from the greeting through the farewell.

Jupyter, VS Code, and Colab prompt you. A headless run, such as `nbconvert`, has no terminal, so the cell replays three lines: Maya's request, "Yes, file the note.", and bye.

`max_steps=8` is the tool-step budget for each of those lines. It is not a limit on how many lines the visit may contain.


In [ ]:
reset_orders()
session = run_customer_chat(
    system_text(),
    demo_utterances=[
        (
            "I'm Maya. Recommend one jar I can eat, put one of those jars in a cart "
            "to ship to Ohio, check that cart, and file an order note for me. "
            "Do not charge my card."
        ),
        "Yes, file the note.",
        "bye",
    ],
    max_steps=8,
    trace_id="guided-purchase",
)

log = "\n".join(session["tool_log"])
check(
    "charge_card" not in log or "denied" in log.lower(),
    "no card was charged",
)
written = list(ORDERS.glob("*.md"))
for path in written:
    body = path.read_text()
    print(body)
    check("charged: no" in body, "the written note does not charge a card")
if session["mode"] == "demo":
    spoken = [
        message["content"]
        for message in session["messages"]
        if message["role"] == "user"
    ]
    check(len(spoken) >= 2, "Maya's request and her yes stayed in one list")
    check(session["turns"] >= 2, "she answered again in the same chat")
    check(bool(session["plan"]), "a plan was written before the tools")
    check("load_skill" in log, "the recommend procedure was loaded")
    check("verify_cart" in log, "the cart was checked")
    check(bool(session["confirmed_tokens"]), "her later message approved the waiting note")
    check(bool(written), "an order note was written after she approved it")
    print("ledger:", cost_summary(session["usage"]))
    check(len(session["spans"]) >= 2, "the purchase has a trace")
